In [139]:
import numpy as np
import matplotlib.pyplot as plt

from sko.GA import GA

from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.sampling.rnd import FloatRandomSampling

In [140]:
# def pond_obj_1(x):
#     objs = obj_fun_1(x[:8])

#     zip_ob = zip(objs, x[8:])

#     return abs(sum([f*w for f,w  in list(zip_ob)]))

# def pond_obj_2(x):
#     objs = obj_fun_2(x[:8])

#     zip_ob = zip(objs, x[8:])

#     return abs(sum([f*w for f,w  in list(zip_ob)]))

In [141]:
fun_1 = [
    lambda x: 0.8*(x[0]**2 + x[0] -  1)*x[2] + 0.12*x[0] + 2.16*x[0] - 0.12,
    lambda x: (1 + x[0]**2)*x[3] + 0.4*x[0]**2 - 1.6*x[0] - 0.4,
    lambda x: (1 + x[0]**2)*x[4] + x[0]**2 - 1,
    lambda x: (1 + x[0]**2)*x[5] + 0.8*(x[0]**2 + x[0] - 1),
    lambda x: x[2]*x[6] - 0.02*x[5] - x[4] - x[2]*x[3] - 0.16*x[3],
    lambda x: x[6]**2 - 2*x[3]*x[6] + x[5]**2 + x[3]**2 - x[1]**2,
    lambda x: x[7] - x[1]*x[2],
    lambda x: 0.0476*x[2]*x[7]**12 + x[2] - 2.104
]

fun_2 = [
    lambda x: x[4] + x[3] - 1.803,
    lambda x: (x[1] + x[2])*x[4] + 6.19116*x[3] - 1.803*(1.497 + 0.035),
    lambda x: x[5] + x[3] - 0.328,
    lambda x: 0.28801*x[5] - x[1]*x[2]*x[4],
    lambda x: (-6.19116*x[0] + x[0]*x[2] + x[1]*x[4] - x[2]*x[4])*x[5] + x[0]*x[2]*x[4],
    lambda x: 1.571*x[6] + x[3] - 1.803,
    lambda x: x[7] - 0.000856*x[6]**2,
    lambda x: (x[4] - x[0])*x[8] - x[0]*x[4],
    lambda x: x[8] - 377*x[1]*x[7]
]


In [142]:
def eval_fun(fun_list, x):
    return [f(x) for f in fun_list]

In [143]:
lb_1 = [-3, -1, -2, -1, -1, -0.5, -1.5, -1.5]
ub_1 = [1, 1, 2, 1, 1, 0.5, 1.5, 1.5]

lb_2 = [-0.5, -1, -1, -1, 1, -1, -1, 0, -1]
ub_2 = [0.5, 1, 1, 1, 2, 1, 1, 1, 1]

In [144]:
def error_moo(result):
    return [sum(x ** 2) for x in result]

In [145]:
class Problema1(ElementwiseProblem):

    def __init__(self):
        super().__init__(
            n_var=8,      
            n_obj=8,      
            xl=lb_1,
            xu=ub_1
        )

    def _evaluate(self, x, out, *args, **kwargs):

        out["F"] = [abs(f(x)) for f in fun_1]

In [146]:
class Problema2(ElementwiseProblem):

    def __init__(self):
        super().__init__(
            n_var=9,      
            n_obj=9,      
            xl=lb_2,
            xu=ub_2
        )

    def _evaluate(self, x, out, *args, **kwargs):

        out["F"] = [abs(f(x)) for f in fun_2]

In [147]:
problem1 = Problema1()
problem2 = Problema2()

algorithm = NSGA2(
    pop_size=100,

    sampling=FloatRandomSampling(),

    crossover=SBX(
        prob=0.6,
        eta=15
    ),

    mutation=PM(
        prob=1.0,
        prob_var = 1 / problem1.n_var,
        eta=20
    ),

    eliminate_duplicates=True
)

result1 = minimize(
    problem1,
    algorithm,
    termination=("n_gen", 1500),
    verbose=False,
    save_history=True
)

# result2 = minimize(
#     problem2,
#     algorithm,
#     termination=("n_gen", 1500),
#     seed=42,
#     verbose=False
# )

In [148]:
# ga_test = GA(
#     func=pond_obj_1,
#     n_dim=16,
#     size_pop=100,
#     max_iter=1000,
#     lb=lb_1 + [-1]*8,
#     ub=ub_1 + [1]*8
# )

In [149]:
# historico = []
# for n in range(30):
#     ga = GA(
#     func=lambda x: fun_1(x)[0],
#     n_dim=8,
#     size_pop=100,
#     max_iter=1000,
#     lb=lb_1,
#     ub=ub_1,
#     constraint_eq=fun_1[1:]
# )
    
#     best_x, best_y = ga.run()
#     historico.append(best_x)

In [150]:
def encontrar_melhor_iteracao_com_escalas(historico_x, lista_funcoes, lb, ub, num_amostras_escala=10000):
    parametros_aleatorios = np.random.uniform(lb, ub, size=(num_amostras_escala, len(lb)))

    escalas = [np.std([f(param) for param in parametros_aleatorios]) for f in lista_funcoes]

    resultados_best_params = np.array([[f(param) for f in lista_funcoes] for param in historico_x])

    resultados_normalizados = resultados_best_params/escalas

    piores_resultados = [np.max(np.abs(r)) for r in resultados_normalizados]

    return historico_x[np.argmin(piores_resultados)]
    

In [151]:
# eval_fun(fun_1, encontrar_melhor_iteracao_com_escalas(historico, fun_1, lb_1, ub_1))

In [152]:
eval_fun(fun_1, encontrar_melhor_iteracao_com_escalas(result1.F, fun_1, lb_1, ub_1))

[np.float64(2.91665828316227),
 np.float64(-1.6276729757811674),
 np.float64(4.26620529969487),
 np.float64(0.9158153119881897),
 np.float64(-1.7130108924514624),
 np.float64(-0.3745260814895366),
 np.float64(-0.487908088290122),
 np.float64(-1.3401631640130045)]

In [155]:
for p in result1.F:
    print(sum(np.abs(eval_fun(fun_1, p))))

4974835.409984434
416.5836911901562
11.169198680131213
13.641960096870624
219322.16302370568
8226023644837.63
3759.337217817614
24156722921054.133
343.8859409046467
460.39572754154693
274.40846721492346
63.5110429841605
19750840998.31595
133.6260545633993
9164705507746.201
257.45296201713666
47947162.13181585
8028.3037958868135
6027631.450926344
27835014577.49484
729978889.2582096
3747952.155752171
5413719233459.337
524.6522487902484
50.31118213166045
287.97574117172144
114051522252848.56
8947.46411586849
63.247190940383156
7639903.523310302
29037356.32664903
247532374.8162551
2112244574.5390127
2004196445629.6096
11423.584995826232
332224360900.64087
141.08076176625804
344292155311.241
113611170.56636223
45108352686034.086
3774552037691.5093
79.50595760432604
126723356161.63814
16126673.183928728
11992844040791.742
1194608790999.9846
37082848939.51977
503.48929129560844
307711850983.0009
116238371061.58646
255.33919857137693
732.4755039933183
122.90214312970988
4018254.8441456486
1833